## Mesure de l'observabilité : 
Vecteur X : X = [θ1 - θ2, ω1 - ω0, ω2 - ω0, Tm1, Tm2, N]

Vecteur d'entrées U : U = [PG1, PG2, PL1, PL2]   

Vecteur de sortie Y : Y = [F12, F21, PG1, PG2, PL1, PL2]

$$\dot{X} = AX + BU $$
$$Y = CX + DU$$

In [55]:
using LinearAlgebra

## Paramètres de départ

In [56]:
KL = 3064 # 1 ligne
ω0 = 100*pi
P01 = 600
P02 = 400
Pmin1 = 0
Pmin2 = 0
Pmax1 = 1000
Pmax2 = 600
Pr1 = 100
Pr2 = 50
J1 = 0.4
J2 = 0.1
D1 = 0.04
D2 = 0.02
α1 = 100
α2 = 100
β1 = 2000
β2 = 2000
Ks =0.05;

### On calcule la dérivée temporelle de X

$$\dot{\theta}_1 - \dot{\theta}_2 = \omega_1 - \omega_2$$

$$J_1 \dot{\omega}_1 = T_1^m - \frac{P_1^G}{\omega_1} - D_1(\omega_1 - \omega_0)$$

$$J_2 \dot{\omega}_2 = T_2^m - \frac{P_2^G}{\omega_2} - D_2(\omega_2 - \omega_0)$$

$$\dot{T}_1^m = -\alpha_1(P_1^m - P_1^c) - \beta_1(\omega_1 - \omega_0)$$

$$\dot{T}_2^m = -\alpha_2(P_2^m - P_2^c) - \beta_2(\omega_2 - \omega_0)$$

$$\dot{N} = K_s\left(\frac{J_1\omega_1 + J_2\omega_2}{J_1+J_2} - \omega_0\right)$$

### On linéarise en considérant ω1≈ω2≈ω0 et P1c≈P01+NPr1, P2c≈P02+NPr2



In [57]:
A = [
    0   1       -1  0   0   0;
    0   -D1/J1  0   1/J1    0   0;
    0   0   -D2/J2  0   1/J2    0;
    0   -β1     0   (-α1 *ω0)   0   -α1*Pr1;
    0   0       -β2     0   (-α2 *ω0)   -α2*Pr2;
    0  (Ks * J1)/(J1 + J2)   (Ks * J2)/(J1 + J2)  0    0   0
    ]

6×6 Matrix{Float64}:
 0.0      1.0      -1.0        0.0       0.0       0.0
 0.0     -0.1       0.0        2.5       0.0       0.0
 0.0      0.0      -0.2        0.0      10.0       0.0
 0.0  -2000.0       0.0   -31415.9       0.0  -10000.0
 0.0      0.0   -2000.0        0.0  -31415.9   -5000.0
 0.0      0.04      0.01       0.0       0.0       0.0

In [58]:
rank(A)

5

## Matrice C

In [59]:
C = [
    KL 0 0 0 0 0;
    -KL 0 0 0 0 0;
    KL 0 0 0 0 0;
    -KL 0 0 0 0 0;
    0 0 0 0 0 0;
    0 0 0 0 0 0;
    ]

6×6 Matrix{Int64}:
  3064  0  0  0  0  0
 -3064  0  0  0  0  0
  3064  0  0  0  0  0
 -3064  0  0  0  0  0
     0  0  0  0  0  0
     0  0  0  0  0  0

Matrice O = [C CA CA² ... CA^5]^T

In [60]:
O = vcat([C * (A^k) for k in 0:5]...)   # concaténation horizontale

36×6 Matrix{Float64}:
  3064.0      0.0             0.0          0.0          0.0        0.0
 -3064.0      0.0             0.0          0.0          0.0        0.0
  3064.0      0.0             0.0          0.0          0.0        0.0
 -3064.0      0.0             0.0          0.0          0.0        0.0
     0.0      0.0             0.0          0.0          0.0        0.0
     0.0      0.0             0.0          0.0          0.0        0.0
     0.0   3064.0         -3064.0          0.0          0.0        0.0
     0.0  -3064.0          3064.0          0.0          0.0        0.0
     0.0   3064.0         -3064.0          0.0          0.0        0.0
     0.0  -3064.0          3064.0          0.0          0.0        0.0
     ⋮                                                             ⋮
     0.0     -4.81298e11      1.92519e12  -7.5601e12    3.024e13   2.40648e12
     0.0      0.0             0.0          0.0          0.0        0.0
     0.0      0.0             0.0          0.0    

Calcul du rang de O

In [61]:
rank(O)

5

In [ ]:

n_non_obs = 6 - rank(O)

# ============================================================================
# 5. TROUVER LE NOYAU (NULL SPACE) DE O
# ============================================================================

if n_non_obs > 0
    println("\n" * "="^60)
    println("VECTEURS NON OBSERVABLES")
    println("="^60)
    
    # Utiliser SVD pour trouver le null space
    U, S, Vt = svd(O, full=true)
    
    # Les dernières colonnes de V correspondent au noyau
    null_space = Vt[end-(n_non_obs-1):end, :]'  # Transpose pour avoir les vecteurs en colonnes
    
    println("\nVecteurs non observables (colonnes) :")
    display(null_space)
    
    # Vérification : O @ null_space devrait être ~0
    residual = O * null_space
    println("\nVérification : ||O @ null_space|| (devrait être ~0) :")
    println("Max norm : $(maximum(abs.(residual)))")
    println("Mean norm : $(mean(abs.(residual)))")
end

# ============================================================================
# 6. ANALYSER LES VALEURS PROPRES DE A
# ============================================================================

println("\n" * "="^60)
println("VALEURS PROPRES DE A")
println("="^60)

eigenvals_A = eigvals(A)
println("\nValeurs propres (λ):")
for (i, λ) in enumerate(eigenvals_A)
    if isreal(λ)
        re = real(λ)
        stable = re < -1e-10 ? "STABLE ✓" : "INSTABLE ✗"
        println("λ_$i = $(round(re, digits=6))  →  $stable")
    else
        re = real(λ)
        im = imag(λ)
        stable = re < -1e-10 ? "STABLE ✓" : "INSTABLE ✗"
        println("λ_$i = $(round(re, digits=6)) + $(round(im, digits=6))i  →  $stable")
    end
end

max_real_part = maximum(real.(eigenvals_A))
println("\nMax(Re(λ)) = $(round(max_real_part, digits=6))")

# ============================================================================
# 7. CONCLURE SUR DETECTABILITÉ
# ============================================================================

println("\n" * "="^60)
println("CONCLUSION")
println("="^60)

observable = rank_O == 6
detectable = (max_real_part < -1e-10) || (rank_O == 6)

# Vérifier si les modes instables sont observables
if !observable && n_non_obs > 0 && !isempty(eigenvals_A)
    unstable_eigenvals = eigenvals_A[real.(eigenvals_A) .> -1e-10]
    if isempty(unstable_eigenvals)
        detectable = true
    end
end

println("\nObservable ? $(observable ? "OUI ✓" : "NON ✗")")
println("Détectable ? $(detectable ? "OUI ✓" : "NON ✗")")
println("Modes non observables stables ? $(max_real_part < -1e-10 ? "OUI ✓" : "À VÉRIFIER ⚠")")

if !observable && detectable
    println("\n→ Système NON observable mais DÉTECTABLE")
    println("→ Peux utiliser Kalman/Observateur + retour d'état")
    println("→ Les $(n_non_obs) direction(s) non observable(s) est(sont) stable(s)")
elseif observable
    println("\n→ Système OBSERVABLE : tous les états reconstruits")
elseif !detectable
    println("\n⚠ ATTENTION : Système NON observable ET NON détectable")
    println("→ Il y a des modes instables non observables !")
    println("→ Il faut ajouter des mesures")
end

# ============================================================================
# 8. (OPTIONNEL) DÉCOMPOSITION OBSERVABLE / NON OBSERVABLE
# ============================================================================

if n_non_obs > 0 && n_non_obs < 6
    println("\n" * "="^60)
    println("DÉCOMPOSITION OBSERVABLE / NON OBSERVABLE")
    println("="^60)
    
    # Normaliser les vecteurs non observables
    V_u = copy(null_space)
    for i in 1:size(V_u, 2)
        V_u[:, i] ./= norm(V_u[:, i])
    end
    
    println("\nVecteurs non observables (normalisés) :")
    display(V_u)
    
    # Compléter avec vecteurs observables via QR
    A_padded = [V_u Matrix(I, 6, 6-n_non_obs)]
    Q, R = qr(A_padded)
    T = Matrix(Q)
    
    # Transformer les matrices
    T_inv = inv(T)
    A_tilde = T_inv * A * T
    C_tilde = C * T
    
    println("\nMatrice A dans la base (observable | non-observable) :")
    display(round.(A_tilde, digits=4))
    
    println("\nMatrice C transformée (devrait avoir 0 à droite) :")
    display(round.(C_tilde, digits=4))
    
    # Analyser les modes non observables
    idx_nonobs = (6-n_non_obs+1):6
    A_u = A_tilde[idx_nonobs, idx_nonobs]
    eigenvals_u = eigvals(A_u)
    
    println("\nValeurs propres des MODES NON OBSERVABLES :")
    for (i, λ) in enumerate(eigenvals_u)
        re = real(λ)
        stable = re < -1e-10 ? "STABLE ✓" : "INSTABLE ✗"
        println("λ_u$i = $(round(re, digits=6))  →  $stable")
    end
end

println("\n" * "="^60)
println("FIN DE L'ANALYSE")
println("="^60)
